# SL ladder · 03 · Recovering two undocumented conventions

The frozen ladder script reads a linked table built on the Linux box. Two choices inside that build
are invisible in the code and absent from the Methods: **how a reporting week maps to a date**, and
**whether the population denominator varies by year**.

Rather than guess, both are recovered by sweeping the candidates against the 3,926 frozen labels.
This notebook is the evidence for the two conventions `sl_02` applies.

## 1 · Setup

In [1]:
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd().parent if Path.cwd().name.startswith("notebooks") else Path.cwd()
DQ   = REPO / "data_quarantine"
ERA  = DQ / "wp5_exposure" / "era5_0p25_window"
CHI  = DQ / "wp5_exposure" / "chirps_window"
OUTD = DQ / "sl_ladder"
OUTD.mkdir(parents=True, exist_ok=True)

o = pd.read_csv(DQ / "wer_srilanka/frozen/wer_dengue_currentweek_rdhs_2018_2025_v2.1-refresh.csv")
o = o.rename(columns={"year": "epi_year", "week": "epi_week"})
we = pd.read_csv(DQ / "wp5_exposure/wp5_buildB_weights_era5_0p25_srilanka_2020.csv")
o = o.merge(we[["geometry_id", "rdhs_name"]].drop_duplicates(), left_on="rdhs", right_on="rdhs_name")

# week_start is rebuilt here from the NAIVE ISO convention so the sweep below is
# genuinely over the raw choice, not over the answer sl_02 already applied.
def iso_monday(y, w):
    try:    return pd.Timestamp(pd.Timestamp.fromisocalendar(int(y), int(w), 1))
    except ValueError: return pd.NaT
wk = o[["epi_year", "epi_week"]].drop_duplicates()
wk["week_start"] = [iso_monday(y, w) for y, w in zip(wk.epi_year, wk.epi_week)]
o = o.merge(wk, on=["epi_year", "epi_week"]).dropna(subset=["week_start"])

fz = pd.read_csv(REPO / "ALT_STATS/frozen/srilanka_matched_pairs.csv", parse_dates=["predictor_week"])
fz = fz[fz.setting == "SriLanka"]
base = o[o.dengue_current_week.notna()][["geometry_id", "week_start", "dengue_current_week", "epi_year"]].copy()
print("frozen labels:", len(fz), "| prevalence", round(fz.outcome.mean(), 6))

frozen labels: 3926 | prevalence 0.336475


## 2 · The label

Incidence four weeks ahead, above the district's own training-period 75th percentile. Built here as a
function of the choices under test, so each can be swept independently.

In [2]:
def build(shift_weeks=0, denom="const", interp="linear", q=0.75):
    f = base.copy()
    f["week_start"] = f["week_start"] + pd.Timedelta(days=7 * shift_weeks)
    den = 1.0 if denom == "const" else f["epi_year"].map(lambda y: 1 + 0.01 * (y - 2018))
    f["inc"] = f["dengue_current_week"] / den
    fut = f[["geometry_id", "week_start", "inc"]].rename(columns={"inc": "inc_future"})
    fut["week_start"] = fut["week_start"] - pd.Timedelta(days=28)
    f = f.merge(fut, on=["geometry_id", "week_start"], how="left")
    f = f[f.inc_future.notna()]
    thr = f[f.week_start.dt.year <= 2022].groupby("geometry_id")["inc"].quantile(q, interpolation=interp)
    f = f.merge(thr.rename("thr").reset_index(), on="geometry_id")
    f["y"] = (f["inc_future"] > f["thr"]).astype(int)
    mg = f.merge(fz[["spatial_unit_id", "predictor_week", "outcome"]],
                 left_on=["geometry_id", "week_start"],
                 right_on=["spatial_unit_id", "predictor_week"], how="inner")
    return (mg.y == mg.outcome).mean(), mg.y.mean(), len(mg)

## 3 · Convention 1 · the week offset

A one-week shift is worth 17 points of label agreement. Nothing else in the sweep comes close, and the
peak is sharp - which is what a genuine calendar convention looks like, as opposed to noise.

In [3]:
for s in [-2, -1, 0, 1, 2]:
    a, p, n = build(shift_weeks=s)
    mark = "  <-- adopted" if s == -1 else ""
    print(f"shift {s:+d} week: agreement {a:.4f}  prevalence {p:.4f}  matched {n}{mark}")

shift -2 week: agreement 0.8287  prevalence 0.3377  matched 3900
shift -1 week: agreement 0.9880  prevalence 0.3383  matched 3926  <-- adopted
shift +0 week: agreement 0.8242  prevalence 0.3426  matched 3926
shift +1 week: agreement 0.8049  prevalence 0.3400  matched 3926
shift +2 week: agreement 0.7761  prevalence 0.3449  matched 3926


## 4 · Convention 2 · the denominator

Holding population constant beats letting it grow. The effect is small but consistent, and it points
the same way as the fact that only 2018-2020 WorldPop is on this machine: the frozen build was not
using year-specific denominators either.

In [4]:
for d in ["const", "growing"]:
    a, p, n = build(shift_weeks=-1, denom=d)
    print(f"denominator {d:8s}: agreement {a:.4f}  prevalence {p:.4f}")

denominator const   : agreement 0.9880  prevalence 0.3383
denominator growing : agreement 0.9860  prevalence 0.3255


## 5 · Threshold construction

With the two conventions fixed, the remaining freedom is how the 75th percentile is interpolated when
it falls between observations. `higher` reproduces the frozen prevalence most closely.

In [5]:
rows = []
for interp in ["linear", "lower", "higher", "nearest", "midpoint"]:
    a, p, n = build(shift_weeks=-1, denom="const", interp=interp)
    rows.append({"interpolation": interp, "agreement": round(a, 4), "prevalence": round(p, 4)})
r = pd.DataFrame(rows).sort_values("agreement", ascending=False)
print(r.to_string(index=False))
print(f"\nfrozen prevalence: {fz.outcome.mean():.4f}")

interpolation  agreement  prevalence
       linear     0.9880      0.3383
        lower     0.9880      0.3383
      nearest     0.9880      0.3383
     midpoint     0.9880      0.3383
       higher     0.9878      0.3365

frozen prevalence: 0.3365


## 6 · Where the residual disagreement sits

98.8% agreement leaves ~47 district-weeks. They are not spread evenly: they concentrate in a handful
of districts and grow with distance from the training period. That is the signature of rows sitting
close to their threshold, where any small difference in the case series flips the label - not of a
systematic error still hiding in the construction.

In [6]:
f = base.copy()
f["week_start"] = f["week_start"] - pd.Timedelta(days=7)
f["inc"] = f["dengue_current_week"]
fut = f[["geometry_id", "week_start", "inc"]].rename(columns={"inc": "inc_future"})
fut["week_start"] = fut["week_start"] - pd.Timedelta(days=28)
f = f.merge(fut, on=["geometry_id", "week_start"], how="left"); f = f[f.inc_future.notna()]
thr = f[f.week_start.dt.year <= 2022].groupby("geometry_id")["inc"].quantile(0.75, interpolation="higher")
f = f.merge(thr.rename("thr").reset_index(), on="geometry_id")
f["y"] = (f["inc_future"] > f["thr"]).astype(int)
mg = f.merge(fz[["spatial_unit_id", "predictor_week", "outcome"]],
             left_on=["geometry_id", "week_start"], right_on=["spatial_unit_id", "predictor_week"])
mg["bad"] = mg.y != mg.outcome
print("disagreements:", int(mg.bad.sum()), "of", len(mg))
print("\nby year:"); print(mg.groupby(mg.predictor_week.dt.year).bad.agg(["sum", "size"]).to_string())
print("\nworst districts:"); print(mg.groupby("geometry_id").bad.sum().sort_values(ascending=False).head(5).to_string())
near = (mg.inc_future - mg.thr).abs() <= mg.thr * 0.15
print(f"\nshare of disagreements within 15% of threshold: {mg.bad[near].sum()/max(mg.bad.sum(),1):.2f}")

disagreements: 48 of 3926

by year:
                sum  size
predictor_week           
2023             15  1352
2024             12  1378
2025             21  1196

worst districts:
geometry_id
LK91    10
LK32     8
LK11     5
LK22     5
LK92     4

share of disagreements within 15% of threshold: 1.00
